# Bounding H<sup>2</sup>MM

Let's get our obligitory imports in order, and we'll load the 3 detector data as well.

In [1]:
import os
import numpy as np
from matplotlib import pyplot as plt

import H2MM_C as hm

# new in v 2.3, use SQUAREM projection acceleration
hm.optimization_limits.squarem = True
##

def load_txtdata(filename):
    """
    load the space-separated bursts file
    """
    color = list()
    times = list()
    with open(filename,'r') as f:
        for i, line in enumerate(f):
            if i % 2 == 0:
                times.append(np.array([int(x) for x in line.split()],dtype=np.int64))
            else:
                color.append(np.array([int(x) for x in line.split()],dtype=np.uint8))
    return color, times

color3, times3 = load_txtdata('sample_data_3det.txt')

## Built in Limits

Sometimes we may want to restrain what values are possible in the H<sup>2</sup>MM model, for instance to keep transition rates within reasonable values, or because you know something about how the emmission probability matrix (`obs`) should behave.

This is expected to happen most often (but not exclusively) when some experimental parameter is periodic, but not important to the data. For instance in $\mu$sALEX experiments, the laser alternation period causes donor and acceptor excitation photons, which arrive in separate streams to alternate (and thus transition) perfectly periodically, yet that has no bearing on transitions between our system. Thus transition rates close to the rate of laser alternation are likely to be artifacts, and thus we want to exclude them. For $\mu sALEX$ experiments, we find this still is not enough.

To define some bounds, we need to define the bounds, this is done using the `hm.h2mm_limits` object, which we pass into the `hm.EM_H2MM_C()` function through the keyword argument `bounds`, and we also need to supply a method string to another keyword argument, `bounds_func`.

#### Let's see an example

In [2]:
alt_period = 4000 # a fake alternation period
us_bounds = hm.h2mm_limits(max_trans = 1/(alt_period))

prior = np.array([1/4, 1/4, 1/4, 1/4])
trans = np.array([[1-3e-6, 1e-6, 1e-6, 1e-6],
                  [1e-6, 1-3e-6, 1e-6, 1e-6],
                  [1e-6, 1e-6, 1-3e-6, 1e-6],
                  [1e-6, 1e-6, 1e-6, 1-3e-6]])
obs = np.array([[0.4,0.4,0.2],
                [0.3,0.1,0.6],
                [0.2,0.4,0.4],
                [0.1,0.1,0.8]])

imodel_4s3d = hm.h2mm_model(prior, trans, obs)

us_opt_model4 = hm.EM_H2MM_C(imodel_4s3d, color3, times3, bounds_func='revert', bounds=us_bounds)
us_opt_model4

The model converged after 442 iterations


nstate: 4, ndet: 3, nphot: 436084, niter: 442, loglik: -408203.01780759863 converged state: 0x126
prior:
0.1974226257040034, 0.5611285905398735, 0.24144878375612308, 1.8439838186141076e-52
trans:
0.9999562426522474, 2.620898295596973e-05, 1.8189495308749981e-06, 1.57294152656387e-05
7.049878430372619e-06, 0.9999698845702242, 6.991336035823985e-06, 1.6074215309619342e-05
1.2716823436474892e-06, 1.7388146198086606e-05, 0.9999781791402766, 3.16103118162608e-06
1.730045723261817e-05, 0.00011453032421588805, 8.076550778759318e-06, 0.9998600926677728
obs:
0.8495291875590435, 0.07564774490030433, 0.07482306754065218
0.47168654842071517, 0.09134404483218894, 0.4369694067470959
0.14909984316802577, 0.31276921689272524, 0.538130939939249
0.1508474048508609, 0.07681327656453163, 0.7723393185846076

So, what did we just do? The `hm.h2mm_limits` object `us_bounds` prevents any value (off the diagonal) of the **transition probability** matrix (`.trans`) from ever being larger (i.e. faster transition rate) than `1/(4000)`. 

#### Bounds process

When you use a bounds method, each iteration goes through the following steps:
1. Calculate *loglikelihood* and new model
2. Check if the **model** converged
3. Analyze the **new model**, and correct if necessary
    1. Check if any values are smaller or larger than a pre-set minimum or maximum
    2. If values are out of bounds, apply correction, method defined by argument passed to `bounds_func`
4. Repeat optimization (back to step 1)

The inputs to `hm.h2mm_limits` are all keyword argumetns, and come in the form of `min/max_[array]` where `[array]` is `prior`, `trans` or `obs`, and specify the minimum and maximum values in the respective array.
Specifying as a float will set the value for all states, and thus the created `hm.h2mm_limits` object can be used for models with any model, while values can be specified as arrays, where each element sets the min/max of the value at that position in the given array of the model.

#### `bounds_func`

As mentioned in the above outline, the bounding process needs to choose how to correct the way in which a model value that is out of bound is corrected.
There are 3 options:

1. `minmax` shallowest correction, sets the value to its minimum or maximum
2. `revert` prefered method, sets the value to the value in the previous model
3. `revert_old` a more extreme form of `revert` which goes to the model before the last in the optimization, and sets the value to that.

### Using `hm.factory_h2mm_model()` with bounds

You will note in the previous example, we specified the `hm.h2mm_model` explicitly, instead of using `hm.factory_h2mm_model()`. 
This is because it is possible that the `hm.factory_h2mm_model()` could create an initial model that contains out of bounds values, which could result in odd behavior during optimization.

There is a way around this, you can give the `hm.h2mm_limits` object to `hm.factory_h2mm_model()` through the keyword argument `bounds`, and the function will automatically ensure the model is with bounds:

> See the full documentation to see full list of options for customizing the `hm.factory_h2mm_model()` function's output

In [3]:
us_bounds = hm.h2mm_limits(max_trans = 1/4000)
# make factory_h2mm_model make a model within bounds
imodel = hm.factory_h2mm_model(3,3, bounds=us_bounds)
us_model = hm.EM_H2MM_C(imodel, color3, times3, bounds=us_bounds, bounds_func='revert')

The model converged after 108 iterations


## Custom Bounds

Finally, it is possible to supply a custom bounding function to `bounds_func`.

> **Note**
>
> This feature was designed to allow the user to handle things/circumstances
> that the writers of H2MM_C had not anticipated.
> Therefore this example is very simple, and does not show a useful method.

This function must is called at the end of the optimization loop, after a given model's loglik has been 
calculated (and the standard H<sup>2</sup>MM next model for the next iteration produced).

This function takes the signature 
`bounds_func(new:h2mm_model, current:h2mm_model, old:h2mm_model, *bounds, **bounds_func)->h2mm_model|int|tuple[h2mm_model,int]`

`new`, `current` and `old` are the `h2mm_model`s of the current iteration.

1. `new` is the model suggested/produced by the current iteration
2. `current` is the model whose loglik was just computed
3. `old` is the model computed in the previous iteration

These are always supplied each iteration. `bounds` and `bounds_kwargs` come from the
identically named keyword arguments in `EM_H2MM_C`. `bounds` by default is `None`, which 
is internally converted to a 0-size (empty) `tuple`, likewise `bounds_kwargs` is by default `None`
and is internally converted into an empty dict.

The return value can either or both specify
1. The "bounded" `new` model
2. If the optimization has converged

If only the `new` model is specified, convergence will be determined like all other optimizaztions,
by the difference in loglik of `current` and `new`. 

**However,** if the bounds function returns a value specifying if the model has converged, then
`EM_H2MM_C` will **not** separately check if the optimization has converged.

> **Note**
> `max_iter` and `max_time` are enforced separetely from `bounds_func`. 

If specifying the converged state, this can be either a `bool` or 0, 1, 2.

As a `bool`, `True` indicates that the optimization has converged, and thus
can stop, the `old` model will be returned as the "optimal" model. `False`
will allow the optimization to proceed using the `new` model.

If specifying as `0` is equivalent to `False`, `1` to `True`, and `2` will return
the 'current' model as the optimal model.

If both the `new` model and converged state are specified, this must be done by returning a 2-tuple
of `(new, converged_state)`.

> **Warning**
> Make sure the model you return makes sense, otherwise the optimization will proceed unpredictably.
> Think of this as the “gloves off” approach, you might have a very powerful new method, or you might
> get something meaningless depending on how you code it. That’s your responsibility.

Below is a function that that re-implements the behavior of `"minmax"` but now the limits
normally specified with a `h2mm_limits` object supplied to `bounds` are replaced with kwargs:

In [4]:
def minmax_py(new, current, old, converged_min=1e-9, 
                  min_prior=None, max_prior=None,
                  min_trans=None, max_trans=None, 
                  min_obs=None, max_obs=None):
    # bounding of trans matrix
    if min_trans is not None or max_trans is not None:
        trans = new.trans
        idxs = np.arange(new.nstate)
        if isinstance(min_trans, float):
            trans[trans < min_trans*(~np.eye(new.nstate, dtype=np.bool_))] = min_trans
        elif isinstance(min_trans, np.ndarray):
            mask = trans < min_trans
            trans[mask] = min_trans[mask]
        if isinstance(max_trans, float):
            trans[trans > max_trans*(~np.eye(new.nstate, dtype=np.bool_))] = max_trans
        elif isinstance(max_trans, np.ndarray):
            mask = trans > max_trans
            trans[mask] = max_trans[mask]
        for i in range(trans.shape[0]):
            trans[i,i] = 1.0 - trans[i, idxs!=i].sum()
        new.trans = trans
    # bounding of obs matrix
    if min_obs is not None or max_obs is not None:
        obs = new.obs
        if min_obs is not None:
            minmask = obs < min_obs
            obs[minmask] = min_obs[minmask]
        else:
            minmask = np.zeros(obs.shape, dtype=np.bool_)
        if max_obs is not None:
            maxmask = obs > max_obs
            obs[maxmask] = max_obs[maxmask]
        else:
            maxmask = np.zeros(obs.shape, dtype=np.bool_)
        obsmask = minmask | maxmask
        for i in range(obs.shape[0]):
            obs[i,~obsmask[i,:]] += (1-obs[i,:].sum()) / (~obsmask).sum()
        new.obs = obs
    if min_prior is not None or max_prior is not None:
        prior = new.prior
        if min_prior is not None:
            minpmask = prior < min_prior
            prior[minpmask] = min_prior[minpmask]
        else:
            minpmask = np.zeros(new.nstate, base=np.bool_)
        if max_prior is not None:
            maxpmask = prior > max_prior
            prior[maxpmask] = max_prior[maxpmask]
        else:
            maxpmask = np.zeros(new.nstate, base=np.bool_)
        pmask = minpmask | maxpmask
        prior[~pmask] += (1-prior.sum()) / (~pmask).sum()
        new.prior = prior
    return new

In [5]:
prior = np.array([1/4, 1/4, 1/4, 1/4])
trans = np.array([[1-3e-6, 1e-6, 1e-6, 1e-6],
                  [1e-6, 1-3e-6, 1e-6, 1e-6],
                  [1e-6, 1e-6, 1-3e-6, 1e-6],
                  [1e-6, 1e-6, 1e-6, 1-3e-6]])
obs = np.array([[0.09,0.01,0.9],
                [0.3,0.1,0.6],
                [0.2,0.4,0.4],
                [0.1,0.1,0.8]])

imodel4s3d = hm.h2mm_model(prior, trans, obs)
us_opt_model4 = hm.EM_H2MM_C(imodel_4s3d, color3, times3, bounds_func=minmax_py, bounds_kwargs=dict(max_trans=1e-4))
us_opt_model4

The model converged after 220 iterations


nstate: 4, ndet: 3, nphot: 436084, niter: 220, loglik: -408204.5620664333 converged state: 0x127
prior:
0.20187268615192835, 0.5555114883353087, 0.24261582551276284, 7.847393954744439e-22
trans:
0.9999561166857909, 2.578067189755747e-05, 1.80782273056363e-06, 1.629481958114899e-05
6.756375458554776e-06, 0.9999721939616957, 6.94983018006591e-06, 1.409983266578162e-05
1.2573142610843664e-06, 1.7601645809611314e-05, 0.9999780804968852, 3.060543044090418e-06
1.999722158662878e-05, 0.0001, 8.607972915624587e-06, 0.9998713948054977
obs:
0.8487005468493065, 0.07578052867734039, 0.07551892447335296
0.47035303129215555, 0.09123834174080213, 0.43840862696704236
0.1491748897867683, 0.31288640664609546, 0.5379387035671364
0.15244038528130474, 0.0771192824936845, 0.7704403322250107

Now let's re-implement how convergence of the optimization is handled:

In [6]:
def limit_converged(new, current, old, conv_min):
    if current.loglik < old.loglik:
        return 1
    if (current.loglik - old.loglik) < conv_min:
        return 2
    return 0

In [7]:
us_opt_model4 = hm.EM_H2MM_C(imodel_4s3d, color3, times3, bounds_func=limit_converged, bounds=5e-8)
us_opt_model4

The model converged after 377 iterations


nstate: 4, ndet: 3, nphot: 436084, niter: 377, loglik: -408203.0178091877 converged state: 0x27
prior:
0.19742841008312292, 0.5611216006883437, 0.24144998922853325, 9.793475396563891e-44
trans:
0.9999562426511326, 2.6207680435119817e-05, 1.818977952218905e-06, 1.5730690480076083e-05
7.049525708530128e-06, 0.9999698869426987, 6.991349490241465e-06, 1.6072182102566528e-05
1.271678751249718e-06, 1.7388305330058e-05, 0.9999781790512848, 3.160964633835334e-06
1.7303501977375584e-05, 0.00011451998455913256, 8.076752081374793e-06, 0.999860099761382
obs:
0.8495280206269593, 0.0756479273278566, 0.0748240520451842
0.47168491833344256, 0.0913439430439943, 0.43697113862256315
0.14909992138007602, 0.31276915727335886, 0.5381309213465651
0.15084605058506348, 0.07681301635449046, 0.7723409330604462

Finally, bellow is an example that re-implements the min-max procedure and checking for convergence:

In [8]:
def minmax_conv_py(new, current, old, conv_min, **kwargs):
    return minmax_py(new, current, old, **kwargs), limit_converged(new, current, old, conv_min)

In [9]:
us_opt_model4 = hm.EM_H2MM_C(imodel_4s3d, color3, times3, bounds_func=minmax_conv_py, bounds=5e-8, bounds_kwargs=dict(max_trans=1e-4))
us_opt_model4

The model converged after 191 iterations


nstate: 4, ndet: 3, nphot: 436084, niter: 191, loglik: -408204.5620671171 converged state: 0x27
prior:
0.2018747146092073, 0.5555096304151803, 0.24261565497561238, 1.119486975274503e-18
trans:
0.9999561169279477, 2.5779109814958056e-05, 1.8079880482389532e-06, 1.6295974189130734e-05
6.756110130198382e-06, 0.9999721942237846, 6.949958484323571e-06, 1.4099707600966716e-05
1.2573494593528597e-06, 1.7601792113678482e-05, 0.9999780806062428, 3.0602521841676388e-06
1.9998772866596023e-05, 0.0001, 8.606756036872259e-06, 0.9998713944710965
obs:
0.8486999090789841, 0.07578059437374979, 0.07551949654726618
0.4703530488443331, 0.09123831648887838, 0.4384086346667886
0.14917480782441572, 0.3128859883955672, 0.537939203780017
0.15244045015929247, 0.07711917080568749, 0.77044037903502

This concludes this tutorial.